<a href="https://colab.research.google.com/github/gerardagustina/breast-cancer-survival-analysis/blob/main/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Preprocessing: GBSG2
Encoding, log-transform, stratified train/test split and standardisation for survival models.

In [1]:
!pip install -q lifelines

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import pandas as pd
import numpy as np
from lifelines.datasets import load_gbsg2

SEED = 42

df = load_gbsg2()
df = df.rename(columns={"cens": "event"})

print(df.shape)
print(df["event"].value_counts())
df.head()

(686, 10)
event
0    387
1    299
Name: count, dtype: int64


,horTh,age,menostat,tsize,tgrade,pnodes,progrec,estrec,time,event
0,no,70,Post,21,II,3,48,66,1814,1
1,yes,56,Post,12,II,7,61,77,2018,1
2,yes,58,Post,35,II,9,52,271,712,1
3,yes,59,Post,17,II,4,60,29,1807,1
4,no,73,Post,35,II,1,26,65,772,1


In [3]:
df["horTh"] = (df["horTh"] == "yes").astype(int)
df["menostat"] = (df["menostat"] == "Post").astype(int)
df = pd.get_dummies(df, columns=["tgrade"], drop_first=True, dtype=int)

df.head()

,horTh,age,menostat,tsize,pnodes,progrec,estrec,time,event,tgrade_II,tgrade_III
0,0,70,1,21,3,48,66,1814,1,1,0
1,1,56,1,12,7,61,77,2018,1,1,0
2,1,58,1,35,9,52,271,712,1,1,0
3,1,59,1,17,4,60,29,1807,1,1,0
4,0,73,1,35,1,26,65,772,1,1,0


In [4]:
for col in ["pnodes", "progrec", "estrec"]:
    df[col] = np.log1p(df[col])

df[["pnodes", "progrec", "estrec"]].describe().round(2)

,pnodes,progrec,estrec
count,686.00,686.00,686.00
mean,1.51,3.35,3.35
std,0.72,1.93,1.84
min,0.69,0.00,0.00
25%,0.69,2.08,2.20
50%,1.39,3.51,3.61
75%,2.08,4.89,4.74
max,3.95,7.78,7.04


In [5]:
FEATURES = ["horTh", "age", "menostat", "tsize", "pnodes",
            "progrec", "estrec", "tgrade_II", "tgrade_III"]

X = df[FEATURES]
y = df[["time", "event"]]

print("X:", X.shape)
print("y:", y.shape)
X.head()

X: (686, 9)
y: (686, 2)


,horTh,age,menostat,tsize,pnodes,progrec,estrec,tgrade_II,tgrade_III
0,0,70,1,21,1.386294,3.891820,4.204693,1,0
1,1,56,1,12,2.079442,4.127134,4.356709,1,0
2,1,58,1,35,2.302585,3.970292,5.605802,1,0
3,1,59,1,17,1.609438,4.110874,3.401197,1,0
4,0,73,1,35,0.693147,3.295837,4.189655,1,0


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y["event"],
    random_state=SEED,
)

print(f"Train: {X_train.shape} | event rate: {y_train['event'].mean():.3f}")
print(f"Test:  {X_test.shape} | event rate: {y_test['event'].mean():.3f}")

Train: (548, 9) | event rate: 0.436
Test:  (138, 9) | event rate: 0.435


In [7]:
from sklearn.preprocessing import StandardScaler

NUM_COLS = ["age", "tsize", "pnodes", "progrec", "estrec"]

X_train = X_train.copy()
X_test = X_test.copy()

scaler = StandardScaler()
X_train[NUM_COLS] = scaler.fit_transform(X_train[NUM_COLS])
X_test[NUM_COLS] = scaler.transform(X_test[NUM_COLS])

print("Train (mean / std):")
print(X_train[NUM_COLS].agg(["mean", "std"]).round(2))
print("\nTest (mean / std):")
print(X_test[NUM_COLS].agg(["mean", "std"]).round(2))

Train (mean / std):
      age  tsize  pnodes  progrec  estrec
mean -0.0    0.0     0.0      0.0     0.0
std   1.0    1.0     1.0      1.0     1.0

Test (mean / std):
       age  tsize  pnodes  progrec  estrec
mean  0.10  -0.05   -0.04     0.10   -0.00
std   0.94   1.02    0.97     0.95    0.97


In [8]:
rule = pd.DataFrame(
    {"mean": scaler.mean_, "std": scaler.scale_},
    index=NUM_COLS,
).round(2)
rule

,mean,std
age,52.85,10.23
tsize,29.47,14.23
pnodes,1.51,0.72
progrec,3.31,1.95
estrec,3.35,1.85


In [9]:
def prepare_data(seed=42, test_size=0.2):
    """Load GBSG2 and return a stratified, standardised train/test split."""
    df = load_gbsg2().rename(columns={"cens": "event"})

    df["horTh"] = (df["horTh"] == "yes").astype(int)
    df["menostat"] = (df["menostat"] == "Post").astype(int)
    df = pd.get_dummies(df, columns=["tgrade"], drop_first=True, dtype=int)

    for col in ["pnodes", "progrec", "estrec"]:
        df[col] = np.log1p(df[col])

    X = df[FEATURES]
    y = df[["time", "event"]]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y["event"], random_state=seed
    )

    X_train, X_test = X_train.copy(), X_test.copy()
    scaler = StandardScaler()
    X_train[NUM_COLS] = scaler.fit_transform(X_train[NUM_COLS])
    X_test[NUM_COLS] = scaler.transform(X_test[NUM_COLS])

    return X_train, X_test, y_train, y_test, scaler

In [10]:
Xtr, Xte, ytr, yte, _ = prepare_data(seed=SEED)

print("X_train identical:", Xtr.equals(X_train))
print("X_test identical: ", Xte.equals(X_test))
print("y_train identical:", ytr.equals(y_train))
print("y_test identical: ", yte.equals(y_test))

X_train identical: True
X_test identical:  True
y_train identical: True
y_test identical:  True
